# 03 — Target (Variável Resposta)

**Objetivo:** construir a variável binária de recompra.

$$Y_{i,t} = \begin{cases} 1, & \text{compra em } (t,\, t+1\text{d}] \\ 0, & \text{caso contrário} \end{cases}$$

**Regras:**
* A janela futura é **exclusiva** para o target — nenhuma feature pode tocá-la
* Todo cliente da população recebe um rótulo (0 ou 1) — sem nulos
* O join com o futuro usa cliente×dia **deduplicado**, para não inflar a população

**Output:** `workspace.ml_bakehouse.population_target`

In [0]:
# ── Configuração central (idem notebook 00) ──
SOURCE  = "samples.bakehouse.sales_transactions"
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"
TARGET_WINDOW = 1

from pyspark.sql import functions as F

population = spark.table(f"{CATALOG}.{SCHEMA}.population_active")
future = (
    spark.table(SOURCE)
    .withColumn("tx_date", F.to_date("dateTime"))
    .select("customerID", "tx_date")
    .distinct()  # 1 linha por cliente × dia — evita duplicar a população no join
)

In [0]:
# Target = 1 se houver compra em [t+1, t+TARGET_WINDOW]
target = (
    population.alias("p")
    .join(
        future.alias("f"),
        on=(F.col("f.customerID") == F.col("p.customerID"))
         & (F.col("f.tx_date").between(
               F.col("p.reference_date") + F.lit(1),
               F.col("p.reference_date") + F.lit(TARGET_WINDOW))),
        how="left",
    )
    .groupBy(F.col("p.reference_date"), F.col("p.customerID"))
    .agg(F.max(F.when(F.col("f.tx_date").isNotNull(), 1).otherwise(0)).alias("target"))
)

# Verificação: mesma contagem da população e sem nulos
print("População:", population.count(), "| Target:", target.count())
print("Nulos no target (esperado 0):", target.filter(F.col("target").isNull()).count())

In [0]:
# Proporção de eventos e estabilidade temporal
estabilidade = (
    target.groupBy("reference_date")
    .agg(
        F.count("*").alias("clientes"),
        F.sum("target").alias("eventos"),
        F.round(F.avg("target"), 4).alias("taxa_evento"),
    )
    .orderBy("reference_date")
)
display(estabilidade)

display(target.agg(
    F.count("*").alias("linhas"),
    F.round(F.avg("target"), 4).alias("taxa_evento_global"),
))

In [0]:
# Gravação
(target.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.population_target"))

print(f"Salvo: {CATALOG}.{SCHEMA}.population_target")
display(spark.table(f"{CATALOG}.{SCHEMA}.population_target").limit(10))

## Conclusões

* Taxa de evento esperada: **~38–45%** por snapshot — classes bem balanceadas.
* A taxa deve ser estável entre snapshots — flutuação forte indicaria target ruidoso.
* **Próximo passo:** `04_eda_i` — análise exploratória da população e da variável resposta.